<a href="https://colab.research.google.com/github/mrpunith-1920/gnn-fraud-detection/blob/main/02_graph_building.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install torch_geometric

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 15.1 MB/s  0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd, numpy as np, torch

path = '/content/drive/MyDrive/gnn-project/data/'   # use the same path that worked before

features = pd.read_csv(path + 'elliptic_txs_features.csv', header=None)
classes  = pd.read_csv(path + 'elliptic_txs_classes.csv')
edges    = pd.read_csv(path + 'elliptic_txs_edgelist.csv')

MessageError: Error: credential propagation was unsuccessful

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
id_map = {tx: i for i, tx in enumerate(features[0])}
print("Number of nodes:", len(id_map))

NameError: name 'features' is not defined

In [3]:
!pip install torch_geometric

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 28.4 MB/s  0:00:00


In [4]:
from google.colab import drive
drive.mount('/content/drive')
path = '/content/drive/MyDrive/gnn-project/data/'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
path = '/content/'

In [7]:
import pandas as pd, numpy as np, torch

features = pd.read_csv(path + 'elliptic_txs_features.csv', header=None)
classes  = pd.read_csv(path + 'elliptic_txs_classes.csv')
edges    = pd.read_csv(path + 'elliptic_txs_edgelist.csv')

print("Features:", features.shape)
print("Classes:", classes.shape)
print("Edges:", edges.shape)

FileNotFoundError: [Errno 2] No such file or directory: '/content/elliptic_txs_features.csv'

In [8]:
import os
print(os.listdir('/content/'))

['.config', 'drive', 'sample_data']


In [9]:
import os
path = '/content/drive/MyDrive/gnn-project/data/'
print(os.listdir(path))


['archive.zip', 'elliptic_bitcoin_dataset']


In [10]:
import os
path = '/content/drive/MyDrive/gnn-project/data/elliptic_bitcoin_dataset/'
print(os.listdir(path))

['elliptic_txs_classes.csv', 'elliptic_txs_edgelist.csv', 'elliptic_txs_features.csv']


In [11]:
import pandas as pd, numpy as np, torch

features = pd.read_csv(path + 'elliptic_txs_features.csv', header=None)
classes  = pd.read_csv(path + 'elliptic_txs_classes.csv')
edges    = pd.read_csv(path + 'elliptic_txs_edgelist.csv')

print("Features:", features.shape)
print("Classes:", classes.shape)
print("Edges:", edges.shape)

Features: (203769, 167)
Classes: (203769, 2)
Edges: (234355, 2)


In [12]:
id_map = {tx: i for i, tx in enumerate(features[0])}
print("Number of nodes:", len(id_map))      # expect 203769

Number of nodes: 203769


In [13]:
src = edges['txId1'].map(id_map)
dst = edges['txId2'].map(id_map)
print("Missing ids:", src.isna().sum(), dst.isna().sum())   # expect 0 0

edge_index = torch.tensor([src.values, dst.values], dtype=torch.long)
print(edge_index.shape)                      # expect [2, 234355]

Missing ids: 0 0
torch.Size([2, 234355])


/tmp/ipykernel_511/3109086020.py:5: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  edge_index = torch.tensor([src.values, dst.values], dtype=torch.long)


In [14]:
x = torch.tensor(features.iloc[:, 2:].values, dtype=torch.float)
timestep = torch.tensor(features[1].values, dtype=torch.long)
print(x.shape)                               # expect [203769, 165]

torch.Size([203769, 165])


In [15]:
y = torch.full((len(id_map),), -1, dtype=torch.long)

cls = classes.copy()
cls['node'] = cls['txId'].map(id_map)
cls['class'] = cls['class'].astype(str)

fraud_nodes = cls.loc[cls['class'] == '1', 'node'].values
legit_nodes = cls.loc[cls['class'] == '2', 'node'].values

y[fraud_nodes] = 1
y[legit_nodes] = 0

print("Fraud:", (y == 1).sum().item())       # about 4545
print("Legit:", (y == 0).sum().item())       # about 42019
print("Unknown:", (y == -1).sum().item())    # about 157205

Fraud: 4545
Legit: 42019
Unknown: 157205


In [16]:
from torch_geometric.data import Data

data = Data(x=x, edge_index=edge_index, y=y)
data.timestep = timestep
print(data)

Data(x=[203769, 165], edge_index=[2, 234355], y=[203769], timestep=[203769])


In [17]:
labeled = data.y != -1
data.train_mask = labeled & (data.timestep <= 34)
data.test_mask  = labeled & (data.timestep > 34)

print("Train nodes:", data.train_mask.sum().item(), "| fraud:", (data.y[data.train_mask] == 1).sum().item())
print("Test nodes: ", data.test_mask.sum().item(),  "| fraud:", (data.y[data.test_mask] == 1).sum().item())

Train nodes: 29894 | fraud: 3462
Test nodes:  16670 | fraud: 1083


In [18]:
torch.save(data, path + 'elliptic_graph.pt')

The fraud rate differs between train and test. It is about 11.6% in train and about 6.5% in test. That is normal for a time-based split, and it’s one reason this setup is more realistic than a random split.
Model performance often drops after time step 43. A well-known event (a dark market shutdown) changed the fraud patterns, so many models do worse on the last steps. If your test scores look lower than expected, this is why, and it’s worth a sentence in your report.